# Column selection — combined (Aroa + Carla)

Merges Aroa's and Carla's independent column-trimming work into one reconciled notebook. Each of us picked columns from the 5 research questions separately; where our lists differed, we took the **union** (kept both sets) rather than picking one arbitrarily — every extra column below is still tied to a specific question, none is scope creep. See the discrepancy notes in each section for exactly what differed and why it was kept.

In [ ]:
import pandas as pd
import yaml

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

## Which tables we're using — and which we're not

(Carla's scoping, confirmed)

**Using**: `application_train`, `previous_application`, `bureau`

**Not using**:
- `application_test` — no `TARGET`, can't use it for anything default-related
- `bureau_balance`, `POS_CASH_balance`, `credit_card_balance` — monthly-level detail tables, too granular for the 5 research questions and too much extra cleaning for the time we have
- `sample_submission` — not a data table, it's a Kaggle competition artifact

---
## `application_train`

In [ ]:
application = pd.read_csv(config['input_data']['application'])
application.shape

### Column selection — reconciling two independent lists

**Common to both of us** (12 columns, both picked these independently — strong signal they're actually needed):
`SK_ID_CURR`, `TARGET`, `CODE_GENDER`, `CNT_CHILDREN`, `AMT_INCOME_TOTAL`, `NAME_INCOME_TYPE`, `NAME_EDUCATION_TYPE`, `NAME_FAMILY_STATUS`, `NAME_HOUSING_TYPE`, `OCCUPATION_TYPE`, `DAYS_BIRTH`, `DAYS_EMPLOYED`

**Carla only** (kept — all relevant to Q1, applicant profile):
- `CNT_FAM_MEMBERS` — household size, complements `CNT_CHILDREN`
- `FLAG_OWN_CAR`, `FLAG_OWN_REALTY` — asset ownership, a reasonable risk signal

**Aroa only** (kept — needed for the financial side of Q1):
- `AMT_CREDIT`, `AMT_ANNUITY` — without these we can't describe the loan itself, only the applicant
- `NAME_CONTRACT_TYPE` — cash vs revolving loan, relevant to risk profile

**Decision: union, not intersection.** Nothing here is decorative — every column ties back to Q1 (applicant profile) or Q3 (loan terms). Checked nulls before keeping the extras: `FLAG_OWN_CAR`/`FLAG_OWN_REALTY` have 0 nulls, `CNT_FAM_MEMBERS` has 2 (out of 307,511 — negligible).

In [ ]:
application_columns = [
    'SK_ID_CURR', 'TARGET',
    'CODE_GENDER', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'NAME_FAMILY_STATUS', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'NAME_CONTRACT_TYPE', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE',
    'NAME_EDUCATION_TYPE', 'NAME_HOUSING_TYPE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED'
]
application = application[application_columns]
application.shape

In [ ]:
application.isnull().sum()

### Save the trimmed `application` table

In [ ]:
application.to_csv(config['output_data']['application'], index=False)

---
## `bureau`

**No discrepancy here** — Aroa and Carla picked the exact same 3 columns independently. Kept minimal on purpose: Q2 ("how does prior credit history relate to default risk?") only needs to know whether a client has an overdue record, not the full history of every prior credit.

In [ ]:
bureau = pd.read_csv(config['input_data']['bureau'])
bureau.shape

In [ ]:
bureau_columns = ['SK_ID_CURR', 'CREDIT_DAY_OVERDUE', 'AMT_CREDIT_SUM_OVERDUE']
bureau = bureau[bureau_columns]
bureau.shape

In [ ]:
bureau.isnull().sum()

### Save the trimmed `bureau` table

In [ ]:
bureau.to_csv(config['output_data']['bureau'], index=False)

---
## `previous_application`

In [ ]:
previous = pd.read_csv(config['input_data']['previous_application'])
previous.shape

### Column selection

**Common to both** (8 columns): `SK_ID_PREV`, `SK_ID_CURR`, `NAME_CONTRACT_STATUS`, `CODE_REJECT_REASON`, `NAME_PORTFOLIO`, `CHANNEL_TYPE`, `NAME_SELLER_INDUSTRY`, `NAME_YIELD_GROUP`

**Carla only** (kept): `NAME_PRODUCT_TYPE` — directly relevant to Q5 ("which products and channels concentrate the risk?"), 0 nulls, 3 categories (`XNA`/`x-sell`/`walk-in`).

Used for: Q3 (returning vs. new client, via `NAME_CONTRACT_STATUS`), Q4 (past rejections, via `NAME_CONTRACT_STATUS` + `CODE_REJECT_REASON`), Q5 (products/channels, via the rest).

In [ ]:
previous_columns = [
    'SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_STATUS', 'CODE_REJECT_REASON',
    'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'NAME_YIELD_GROUP',
    'CHANNEL_TYPE', 'NAME_SELLER_INDUSTRY'
]
previous = previous[previous_columns]
previous.shape

In [ ]:
previous.isnull().sum()

### Save the trimmed `previous_application` table

In [ ]:
previous.to_csv(config['output_data']['previous_application'], index=False)

---
## Which research question needs which table/columns

| Question | Table(s) | Key columns |
|---|---|---|
| Q1 — applicant profiles | `application` | demographics + loan terms (all of the above) |
| Q2 — prior credit history | `bureau` | `CREDIT_DAY_OVERDUE`, `AMT_CREDIT_SUM_OVERDUE` |
| Q3 — returning vs. new client | `previous_application` | `NAME_CONTRACT_STATUS` |
| Q4 — were past rejections right? | `previous_application` | `NAME_CONTRACT_STATUS`, `CODE_REJECT_REASON` |
| Q5 — products & channels | `previous_application` | `NAME_PORTFOLIO`, `NAME_PRODUCT_TYPE`, `NAME_YIELD_GROUP`, `CHANNEL_TYPE`, `NAME_SELLER_INDUSTRY` |

All three trimmed tables are saved to `data/clean/` — next step is loading them into MySQL per the schema in `sql_scripts/create_database.sql`.